# Neo triage agent: Claude + the neo_mcp server

This notebook builds a vulnerability-triage agent with the Anthropic Python SDK.

1. Start `neo_mcp_server.py` as a local **stdio** MCP server (mock mode, so no AWS or network credentials are needed for discovery).
2. Turn its tools into Claude tools with `anthropic.lib.tools.mcp.async_mcp_tool`.
3. Let the SDK's **tool runner** drive the agent loop: Claude calls Neo tools, the runner executes them over MCP and feeds the results back.

**Setup**
- `pip install -e ".[agent]"` in the repo's `.venv`, then select that `.venv` as this notebook's kernel.
- Put `ANTHROPIC_API_KEY=...` in the repo-root `.env` (already git-ignored).

In [2]:
import asyncio
import os
import sys
from pathlib import Path

import anthropic
from anthropic.lib.tools.mcp import async_mcp_tool
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
load_dotenv(REPO_ROOT / ".env")

client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY
MODEL = "claude-opus-5-5"

## 1. Start the neo_mcp server

The server runs as a subprocess of this kernel, using the kernel's own interpreter (the project `.venv`).
A background task holds the stdio connection open, so the session (and the server's in-memory scan store) survives across cells.

In [3]:
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(REPO_ROOT / "neo_mcp" / "neo_mcp_server.py")],
    # Merged over the MCP SDK's safe default environment (PATH etc.).
    # For live mode, set NEO_MOCK_MODE to "false" and pass AWS_PROFILE / GITHUB_TOKEN here.
    env={"NEO_MOCK_MODE": "true"},
)
# Jupyter's sys.stderr has no OS file handle, so send the server's stderr to a log file.
SERVER_LOG = REPO_ROOT / "out" / "neo_mcp_server.log"
SERVER_LOG.parent.mkdir(exist_ok=True)


async def _hold_session(ready: asyncio.Future, stop: asyncio.Event) -> None:
    try:
        with open(SERVER_LOG, "a", encoding="utf-8") as errlog:
            async with stdio_client(server_params, errlog=errlog) as (read, write):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    ready.set_result(session)
                    await stop.wait()
    except BaseException as exc:
        if not ready.done():
            ready.set_exception(exc)
        raise


_stop = asyncio.Event()
_ready = asyncio.get_running_loop().create_future()
_server_task = asyncio.create_task(_hold_session(_ready, _stop))
neo = await asyncio.wait_for(_ready, timeout=60)

mcp_tools = (await neo.list_tools()).tools
tools = [async_mcp_tool(t, neo) for t in mcp_tools]
for t in mcp_tools:
    print(f"{t.name:34} {(t.description or '').strip().splitlines()[0]}")

load_mock_scan                     Load findings from a local mock fixture file into the scan store for testing.
pull_aws_inspector_findings        Pull active CVE findings from AWS Inspector v2 for a specific account and region.
pull_aws_security_hub_findings     Pull active CVE findings aggregated in AWS Security Hub for an account and region.
pull_azure_defender_findings       Pull CVE-level vulnerability findings from Microsoft Defender for Cloud.
pull_gcp_scc_findings              Pull CVE-level vulnerability findings from Google Cloud Security Command Center.
run_nessus_scan                    Export findings from a Nessus scan (most recent completed scan if scan_id_hint omitted).
run_qualys_scan                    Pull active vulnerability detections from Qualys VMDR.
run_rapid7_scan                    Pull vulnerability findings from Rapid7 InsightVM across all assets.
score_findings                     Enrich findings with EPSS + CISA KEV and compute composite risk scores.
cre

## 2. The agent

`ask()` runs one user turn through the tool runner and appends every message (Claude's tool calls *and* the tool results) to `history`, so follow-up questions keep full context.

- Adaptive thinking with `display: "summarized"` shows a short summary of Claude's reasoning.
- `fallbacks="default"` lets the API retry on a fallback model if a request is declined by a safety classifier.

In [4]:
SYSTEM = """You are Neo, a vulnerability-triage agent for an enterprise security team.

Use the Neo tools to discover findings, score them, and explain results. Neo scores findings as
risk = likelihood x impact x exposure_factor x 100, with tiers P1 >= 40, P2 >= 20, P3 >= 8, P4 < 8.
A CISA KEV finding on a public asset is always P1.

The server runs in mock mode. Demo fixtures live at "neo_core/examples/mock_findings_multi_source.json"
(every source) and "neo_core/examples/mock_findings.json" (small AWS set); load one with load_mock_scan,
then score it with score_findings.

Ground every claim in tool output and cite CVE IDs and resource IDs. Only open a remediation PR when
the user explicitly asks for one."""

history: list = []


def _clip(text: str, limit: int = 600) -> str:
    return text if len(text) <= limit else text[:limit] + f" ... [{len(text) - limit} more chars]"


def _show_assistant(message) -> None:
    for block in message.content:
        if block.type == "thinking" and block.thinking:
            print(f"\n[thinking] {_clip(block.thinking, 400)}")
        elif block.type == "text":
            print(f"\n{block.text}")
        elif block.type == "tool_use":
            print(f"\n-> {block.name}({_clip(str(block.input), 300)})")


def _show_tool_results(tool_message) -> None:
    for result in tool_message["content"]:
        content = result.get("content", "")
        text = content if isinstance(content, str) else " ".join(
            c.get("text", "") for c in content if isinstance(c, dict)
        )
        flag = " (error)" if result.get("is_error") else ""
        print(f"<- result{flag}: {_clip(text)}")


async def ask(prompt: str, effort: str = "high"):
    history.append({"role": "user", "content": prompt})
    runner = client.beta.messages.tool_runner(
        model=MODEL,
        max_tokens=16000,
        system=SYSTEM,
        tools=tools,
        messages=list(history),
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": effort},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )
    message = None
    async for message in runner:
        history.append(message.to_param())
        _show_assistant(message)
        tool_results = await runner.generate_tool_call_response()  # cached; the runner reuses it
        if tool_results is not None:
            history.append(tool_results)
            _show_tool_results(tool_results)

    if message.stop_reason == "refusal":
        print(f"\n[declined] {message.stop_details}")
    elif message.stop_reason == "max_tokens":
        print("\n[stopped at max_tokens - ask it to continue]")
    return message

## 3. Run it

In [5]:
_ = await ask(
    "Load the multi-source mock fixture and score every finding. "
    "Give me the tier breakdown and the top 5 risks with one line on why each ranks where it does."
)

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CgVCLEBmEysEeJh9i1DLU'}

Follow-up questions reuse the same history and the same MCP session, so the agent can refer back to the `scan_id` it already scored.

In [ ]:
_ = await ask("Take the highest-risk finding and write a short remediation plan: immediate mitigation, permanent fix, and how to verify it.")

## 4. Shut down

Stops the MCP server subprocess. Run the "Start the neo_mcp server" cell again to reconnect; set `history = []` to start a fresh conversation.

In [ ]:
_stop.set()
await _server_task